# QELM on the charge Q under intermediate Z measurements

Experiment `experiments/QELM_Q/exp_QELM_Q.py` (issue #163), a preparatory noise experiment. The reservoir is the one of `02_QELM_M_analysis.ipynb`: a disordered TFIM `H = h sum_i Z_i + sum_{i<j} J_ij X_i X_j` on the `N` system qubits, and an ancilla `R` entangled with the charge of the system, `(|0>_R |Q1>_S + |1>_R |Q2>_S) / sqrt(2)` with `Q1 = N // 2` and `Q2 = N // 2 - 1`.

For each sample:

1. `R` is measured in Z: the outcome fixes the charge sector of the system, and the label is that charge, `Y = Q` (`Q1` or `Q2`, probability 1/2 each).
2. The system evolves for the total time `t` in `nbr_time_evolution` steps (default 5). **Between** two steps, each system qubit is measured projectively in Z with probability `p` and the state collapses. By default there is no measurement after the last step. With `projective_measurement.final_measurement=true` (`FINAL_VALUES` below) a last round follows the last step too, so the features are read right after a collapse; a measured qubit then has `<Z_k> = ±1`.
3. The features are the final expectation values `<Z_k> = Tr(Z_k rho)` of the system qubits, plus a bias. The outcomes of the intermediate measurements are **not** features: they only disturb the state. The same readout is also fitted, separately, on five other feature sets read from the same final states: `<X_k>`, `<Y_k>`, and the two-site correlators `<X_i X_j>`, `<Y_i Y_j>`, `<Z_i Z_j>` over the pairs `i < j` (last section).

A ridge readout is fitted on `Q` and scored on a test set. The primary metric is the **accuracy**: a prediction is right when it lies on the side of `(Q1 + Q2) / 2` of the true charge.

At `p = 0` nothing is measured and the setup is the standard QELM, one unitary `exp(-i H t)` per sample: this is the baseline. The measurement choices and outcomes use their own random generator, so every `p` sees the same Hamiltonians and labels for a given seed.

What to expect: a Z measurement commutes with the charge `Q = sum_i (1 - Z_i) / 2`, and the `X_i X_j` couplings change `Q` by 0 or 2, so the parity of `Q` is always conserved. At the default field `h = 10 J` the large `Z` term makes `Q` itself almost conserved, and the readout should stay accurate whatever `p`: the measurements change the state that carries `Q`, not `Q`. At `h = J` the couplings move the charge around, the final `<Z_k>` no longer add up to `Q`, and the measurements, which pin the state to random configurations, can make the linear readout lose track of it. Sweeping both fields separates the two cases.

Without a final round, the last unitary step rebuilds the entanglement that the measurements removed, and the half-system entropy of the final state hardly moves with `p`. The final round shows the disruption in the final state itself, at the price of features that are partly outcomes. Both settings are swept and shown in two rows of panels.

**Qubit counts (issue #235).** `N` is the number of system qubits, the ancilla excluded; the simulated state has `N_total = N + 1` qubits. The sizes of this notebook, its defaults and the results quoted in its text were moved to this convention: a result obtained at `N = 6` before issue #235, when `N` counted the ancilla, is quoted here at `N = 5`, the same system.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_q_notebook.py` does for a small sweep). Notation, the same in every notebook: `N` is the number of system qubits, ancilla excluded (issue #235), and `L` the number of samples per run. `H_VALUES` are the transverse fields `TFIM.h`, in units of `TFIM.J = 1`, and `FINAL_VALUES` says whether a measurement round follows the last step (`projective_measurement.final_measurement`). Each sweep goes to this notebook's own folder, `outputs/QELM_Q/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one.

In [ ]:
N_VALUES = os.environ.get("N_VALUES", "3 5")  # system qubits N, ancilla excluded
H_VALUES = os.environ.get("H_VALUES", "10 1")  # TFIM.h, in units of TFIM.J
SEEDS = os.environ.get("SEEDS", "42 43 44 45 46")
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.5 0.7 1.0")
# Without / with a measurement round after the last step.
FINAL_VALUES = os.environ.get("FINAL_VALUES", "false true")
L = os.environ.get("L", "400")  # samples per run
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"

## Baseline check at p = 0

With `nbr_time_evolution = 1` there is no intermediate measurement at all: one unitary `exp(-i H t)` per sample, the standard QELM. At `p = 0` and the default 5 steps, the features must be the same up to rounding (five steps `exp(-i H t / 5)` make `exp(-i H t)`), and so must the labels and the accuracy. The check runs the experiment in this process on a few samples, for the first `N` and the first field of the sweep.

In [ ]:
import importlib.util

import numpy as np

from QELM import load_config

spec = importlib.util.spec_from_file_location(
    "exp_QELM_Q", Path("experiments/QELM_Q/exp_QELM_Q.py")
)
assert spec is not None and spec.loader is not None
exp_QELM_Q = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp_QELM_Q)

check = [
    "models=TFIM",
    f"N={N_VALUES.split()[0]}",
    f"TFIM.h={H_VALUES.split()[0]}",
    "samples.L=40",
]
standard = exp_QELM_Q.run(
    load_config([*check, "projective_measurement.nbr_time_evolution=1"])
)
at_p0 = exp_QELM_Q.run(load_config([*check, "projective_measurement.p=0.0"]))
assert at_p0["n_measured_mean"] == 0
np.testing.assert_array_equal(at_p0["Y"], standard["Y"])
np.testing.assert_allclose(at_p0["X"], standard["X"], atol=1e-10)
assert at_p0["accuracy"] == standard["accuracy"]
print(
    f"p = 0 recovers the standard QELM: max |dX| = "
    f"{np.max(np.abs(at_p0['X'] - standard['X'])):.1e}, "
    f"accuracy {at_p0['accuracy']:.3f} = {standard['accuracy']:.3f}"
)

## Launch the sweep

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "QELM_Q"

if RUN:
    # Create the sweep folder here and hand it to the launcher (OUT_DIR):
    # the cells below read this sweep and no other.
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "N_VALUES": N_VALUES,
        "H_VALUES": H_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "FINAL_VALUES": FINAL_VALUES,
        # The default time t = 10 / J; the last section sweeps it.
        "T_VALUES": "10",
        "L": L,
        "OUT_DIR": str(sweep_dir),
    }
    n_runs = (
        len(N_VALUES.split())
        * len(H_VALUES.split())
        * len(SEEDS.split())
        * len(P_VALUES.split())
        * len(FINAL_VALUES.split())
    )
    # One bar for the whole sweep, one step per run; raises
    # CalledProcessError, with the runs' output, if a run fails.
    run_with_progress(
        ["bash", "experiments/QELM_Q/exp_QELM_Q.sh"], total=n_runs, env=env
    )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

In [ ]:
from QELM.utils import load_sweep, summarize

# (key in metrics.json, label).
METRICS = [
    ("accuracy", "accuracy on Q (test)"),
    ("r2", "R2 of the prediction of Q (test)"),
    ("S_half_mean", "half-system entropy S_half (bits)"),
]
rows = load_sweep(
    sweep_dir,
    keys=(
        ("N", "N"),
        ("h", "TFIM.h"),
        ("final", "projective_measurement.final_measurement"),
        ("seed", "seed"),
        ("p", "projective_measurement.p"),
    ),
)
table = summarize(
    rows,
    by=("N", "h", "final", "p"),
    metrics=[key for key, _ in METRICS]
    + ["accuracy_baseline"]
    + [f"accuracy_{name}" for name in exp_QELM_Q.FEATURE_SETS],
)
print(f"{len(rows)} runs")
print(
    f"{'N':>3} {'h':>5} {'final':>5} {'p':>5} {'runs':>4} "
    + " ".join(f"{key:>20}" for key, _ in METRICS)
)
for r in table:
    values = " ".join(
        f"{r[f'{key}_mean']:>9.3g} ± {r[f'{key}_sem']:<8.2g}" for key, _ in METRICS
    )
    print(
        f"{r['N']:>3} {r['h']:>5g} {str(r['final']):>5} {r['p']:>5.2f} "
        f"{r['n_runs']:>4} {values}"
    )

## Metrics against p

One column per metric and one row per setting of the final round (top: measurements between steps only; bottom: also after the last step), one curve per size `N` and field `h` (solid for the first field, dashed for the others), the mean over the seeds with the standard error as error bars. On the accuracy panel the dotted gray line is chance, 0.5; the majority-class baseline of each run is in `accuracy_baseline` in the table. The point at `p = 0` is the standard QELM baseline checked above.

The figure and the data it shows are saved in the sweep folder, as `accuracy_vs_p.png` and `accuracy_vs_p.json` (`QELM.utils.plots.save_figure`).

In [ ]:
import matplotlib.pyplot as plt

from QELM.utils.plots import (
    color_map,
    draw_curve,
    save_figure,
    sizes,
    subplots,
    sweep_curve,
)

finals = sorted({bool(r["final"]) for r in table})
fig, axes = subplots(
    len(finals),
    len(METRICS),
    figsize=(6.5 * len(METRICS), 5.5 * len(finals)),
    squeeze=False,
)
n_values = sizes(table)
hs = sorted({r["h"] for r in table}, reverse=True)
labels = {(N, h): f"N = {N}, h = {h:g} J" for N in n_values for h in hs}
colors = color_map([f"N = {N}" for N in n_values])
for final, row in zip(finals, axes, strict=True):
    setting = "with a final round" if final else "between steps only"
    for N in n_values:
        for k, h in enumerate(hs):
            rows_h = [r for r in table if r["h"] == h and bool(r["final"]) == final]
            for (key, _), ax in zip(METRICS, row, strict=True):
                curve = sweep_curve(rows_h, N, "p", key)
                draw_curve(
                    ax,
                    curve,
                    colors[f"N = {N}"],
                    label=labels[(N, h)],
                    linestyle="-" if k == 0 else "--",
                )
    for (key, label), ax in zip(METRICS, row, strict=True):
        if key == "accuracy":
            ax.axhline(0.5, color="gray", linestyle=":", label="chance")
        ax.set(
            xlabel="measurement probability p",
            ylabel=label,
            title=f"{label}\n{setting}",
        )
        ax.legend()
fig.suptitle("QELM on the charge Q with intermediate Z measurements")
figure_path = save_figure(fig, sweep_dir, "accuracy_vs_p", data={"table": table})
print(f"figure: {figure_path}")
plt.show()

## Accuracy for each observable

The readout of `Q` fitted on each feature set, one panel per set: the one-site `<X_k>`, `<Y_k>`, `<Z_k>` (top row) and the two-site correlators `<X_i X_j>`, `<Y_i Y_j>`, `<Z_i Z_j>` over the pairs `i < j` of system qubits (bottom row). One figure per setting of the final round, the curves as above.

What to expect. The TFIM `h sum Z + sum J X X` and the Z measurements conserve the parity of `Q`, while `X_k` and `Y_k` flip it: `<X_k> = <Y_k> = 0` exactly, and these two panels sit at the majority-class level. `<Z_k>` and `<Z_i Z_j>` are diagonal in the Z basis, so the Z measurements do not erase what they hold about `Q`. `<X_i X_j>` and `<Y_i Y_j>` keep the parity but are nonzero only through coherences between basis states, which the Z measurements destroy: these are the observables for which the entanglement disruption should cost accuracy.

The figures go to the sweep folder, `accuracy_by_observable_<setting>.png` and `.json`.

In [ ]:
OBSERVABLES = ("X", "Y", "Z", "XX", "YY", "ZZ")
observable_figures = {}
for final in finals:
    setting = "final_round" if final else "between_steps"
    fig_obs, axes_obs = subplots(2, 3, figsize=(19.5, 11), squeeze=False)
    rows_final = [r for r in table if bool(r["final"]) == final]
    for name, ax in zip(OBSERVABLES, axes_obs.flat, strict=True):
        key = f"accuracy_{name}"
        for N in n_values:
            for k, h in enumerate(hs):
                rows_h = [r for r in rows_final if r["h"] == h]
                draw_curve(
                    ax,
                    sweep_curve(rows_h, N, "p", key),
                    colors[f"N = {N}"],
                    label=labels[(N, h)],
                    linestyle="-" if k == 0 else "--",
                )
        ax.axhline(0.5, color="gray", linestyle=":", label="chance")
        ax.set_ylim(0.3, 1.05)
        pair = len(name) == 2
        ax.set(
            xlabel="measurement probability p",
            ylabel="accuracy on Q (test)",
            title=f"<{name[0]}_i {name[1]}_j>" if pair else f"<{name}_k>",
        )
        ax.legend()
    title = "with a final round" if final else "between steps only"
    fig_obs.suptitle(f"Accuracy on Q for each observable, {title}")
    observable_figures[setting] = save_figure(
        fig_obs, sweep_dir, f"accuracy_by_observable_{setting}", data={"table": table}
    )
    print(f"figure: {observable_figures[setting]}")
    plt.show()

## A better readout at N = 5: the evolution time

At `N = 5` the coherence observables `<X_i X_j>` and `<Y_i Y_j>` sit at chance already at `p = 0` with the default time `t = 10 / J`. Two fixes were tried first, at `h = 10 J` and `p = 0`, and neither helps: more samples (`L = 2000`: accuracy 0.53) and the cross correlators `<X_i Y_j>`, `<Y_i X_j>` (set `XY`, accuracy 0.45 alone, 0.54 together with `XX`, `YY`). What helps is a shorter evolution. Each system starts in the uniform superposition of its charge sector, whose two-site coherences depend on `Q`: `<X_i X_j> = <Y_i Y_j> = 2 Q (N - Q) / (N (N - 1))`, i.e. 0.6 for `Q = 2` and 0.4 for `Q = 1` at `N = 5`. The random couplings `J_ij` then dephase these coherences sample by sample, and after `t = 10 / J` little of that difference is left. At `t = 0.5 / J` the readout of `<X_i X_j>, <Y_i Y_j>` recovers `Q` perfectly at `p = 0`.

This section sweeps the total time `t` (`T_VALUES`, in units of `1 / J`) at `N = 5`, `h = 10 J`, measurements between steps only, with the same steps and seeds as above, and plots the accuracy of `<X_i X_j>`, `<Y_i Y_j>`, `<X_i Y_j>` and `<Z_k>` against `p`, one curve per `t`. It runs in its own folder, `outputs/QELM_Q_time/<date_time>/`; `RUN = False` reads back the latest sweep of that folder. The figure goes there, as `accuracy_vs_p_time.png` and `.json`.

In [ ]:
TIME_N = os.environ.get("TIME_N", "6")
T_VALUES = os.environ.get("T_VALUES", "0.5 2 10")  # evolution.t, in units of 1 / J
TIME_EXPERIMENT = "QELM_Q_time"

if RUN:
    time_dir = new_sweep_dir(OUT_ROOT, TIME_EXPERIMENT)
    env = {
        **os.environ,
        "N_VALUES": TIME_N,
        "H_VALUES": "10",
        "T_VALUES": T_VALUES,
        "FINAL_VALUES": "false",
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "L": L,
        "OUT_DIR": str(time_dir),
    }
    n_runs = (
        len(TIME_N.split())
        * len(T_VALUES.split())
        * len(SEEDS.split())
        * len(P_VALUES.split())
    )
    run_with_progress(
        ["bash", "experiments/QELM_Q/exp_QELM_Q.sh"], total=n_runs, env=env
    )
else:
    time_dir = latest_sweep(OUT_ROOT / TIME_EXPERIMENT)
print(f"sweep: {time_dir}")

TIME_OBSERVABLES = ("XX", "YY", "XY", "Z")
time_rows = load_sweep(
    time_dir,
    keys=(
        ("N", "N"),
        ("t", "evolution.t"),
        ("seed", "seed"),
        ("p", "projective_measurement.p"),
    ),
)
time_table = summarize(
    time_rows,
    by=("N", "t", "p"),
    metrics=[f"accuracy_{name}" for name in TIME_OBSERVABLES],
)
print(f"{len(time_rows)} runs")
print(
    f"{'N':>3} {'t':>5} {'p':>5} {'runs':>4} "
    + " ".join(f"{'accuracy_' + name:>20}" for name in TIME_OBSERVABLES)
)
for r in time_table:
    values = " ".join(
        f"{r[f'accuracy_{name}_mean']:>9.3g} ± {r[f'accuracy_{name}_sem']:<8.2g}"
        for name in TIME_OBSERVABLES
    )
    print(f"{r['N']:>3} {r['t']:>5g} {r['p']:>5.2f} {r['n_runs']:>4} {values}")

In [ ]:
time_Ns = sizes(time_table)
ts = sorted({r["t"] for r in time_table})
time_colors = color_map([f"t = {t:g} / J" for t in ts])
fig_t, axes_t = subplots(
    len(time_Ns),
    len(TIME_OBSERVABLES),
    figsize=(6.5 * len(TIME_OBSERVABLES), 5.5 * len(time_Ns)),
    squeeze=False,
)
for N, row in zip(time_Ns, axes_t, strict=True):
    for name, ax in zip(TIME_OBSERVABLES, row, strict=True):
        for t in ts:
            rows_t = [r for r in time_table if r["t"] == t]
            draw_curve(
                ax,
                sweep_curve(rows_t, N, "p", f"accuracy_{name}"),
                time_colors[f"t = {t:g} / J"],
                label=f"t = {t:g} / J",
            )
        ax.axhline(0.5, color="gray", linestyle=":", label="chance")
        ax.set_ylim(0.3, 1.05)
        if name == "XY":
            title = "<X_i Y_j>, <Y_i X_j>"
        elif len(name) == 2:
            title = f"<{name[0]}_i {name[1]}_j>"
        else:
            title = f"<{name}_k>"
        ax.set(
            xlabel="measurement probability p",
            ylabel="accuracy on Q (test)",
            title=f"{title}, N = {N}",
        )
        ax.legend()
fig_t.suptitle("Accuracy on Q against p for several evolution times, h = 10 J")
time_figure = save_figure(
    fig_t, time_dir, "accuracy_vs_p_time", data={"table": time_table}
)
print(f"figure: {time_figure}")
plt.show()